# Steam — Analyse du marché des jeux vidéo
**Contexte :** Ubisoft souhaite lancer un nouveau jeu. On analyse le marché Steam pour comprendre les tendances, les genres porteurs et ce qui fait la popularité d'un jeu.

## Setup & Chargement

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, BooleanType

# Désactive le mode ANSI : les valeurs non parsables retournent null au lieu de lever une exception
spark.conf.set("spark.sql.ansi.enabled", "false")

In [ ]:
PATH = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

df_raw = (
    spark.read
    .option("multiLine", "true")
    .json(PATH)
)

print("Schéma brut :")
df_raw.printSchema()

In [ ]:
# Le JSON a une structure {id, data: {appid, name, developer, ...}}
# On utilise getField() pour extraire les champs imbriqués
df = df_raw.select(
    F.col("data").getField("appid").alias("appid"),
    F.col("data").getField("name").alias("name"),
    F.col("data").getField("developer").alias("developer"),
    F.col("data").getField("publisher").alias("publisher"),
    F.col("data").getField("genre").alias("genre_str"),
    F.col("data").getField("type").alias("type"),
    F.col("data").getField("owners").alias("owners"),
    F.col("data").getField("positive").cast(IntegerType()).alias("positive"),
    F.col("data").getField("negative").cast(IntegerType()).alias("negative"),
    F.col("data").getField("price").cast(DoubleType()).alias("price_cents"),
    F.col("data").getField("discount").cast(IntegerType()).alias("discount"),
    F.col("data").getField("languages").alias("languages"),
    F.col("data").getField("release_date").alias("release_date"),
    F.col("data").getField("required_age").cast(IntegerType()).alias("required_age"),
    F.col("data").getField("platforms").getField("windows").cast(BooleanType()).alias("win"),
    F.col("data").getField("platforms").getField("mac").cast(BooleanType()).alias("mac"),
    F.col("data").getField("platforms").getField("linux").cast(BooleanType()).alias("linux"),
)

print(f"{df.count()} jeux chargés")
display(df.limit(3))

### Préparation des données

In [ ]:
# Typage et calcul des colonnes utiles
df = (
    df
    .withColumn("price_eur", F.col("price_cents") / 100)
    .withColumn("total_avis", F.col("positive") + F.col("negative"))
    .withColumn(
        "positive_ratio",
        F.round(F.col("positive") / F.col("total_avis") * 100, 1)
    )
    .withColumn(
        "release_year",
        F.regexp_extract(F.col("release_date"), r"^(\d{4})", 1).cast(IntegerType())
    )
)

# DataFrame explosé : une ligne par couple jeu/genre
# On utilise explode() pour éclater la colonne genre_str (valeurs séparées par des virgules)
df_genres = (
    df
    .withColumn("genre", F.explode(F.split(F.col("genre_str"), ",")))
    .withColumn("genre", F.trim(F.col("genre")))
    .filter(F.col("genre") != "")
    .filter(F.col("genre").isNotNull())
)

print(f"df principal : {df.count()} jeux")
print(f"df_genres    : {df_genres.count()} lignes (jeux × genres)")

---
## 1. Analyse Macro
### Éditeurs les plus prolifiques

In [ ]:
top_pub = (
    df
    .groupBy("publisher")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
    .withColumnRenamed("count", "Nombre de jeux")
    .withColumnRenamed("publisher", "Publisher")
)

display(top_pub)

**Interprétation :** Big Fish Games domine largement avec plus de 400 jeux — c'est un éditeur spécialisé dans les casual games. SEGA et Square Enix sont les seuls grands studios traditionnels dans ce top 20.

### Jeux les mieux notés

In [ ]:
# Filtrer sur min. 100 avis pour écarter les faux 100%
top_rated = (
    df
    .filter(F.col("total_avis") >= 100)
    .orderBy(F.desc("positive_ratio"))
    .select("name", "publisher", "positive_ratio", "positive", "release_year")
    .limit(15)
)

display(top_rated)

In [ ]:
# Top par volume d'avis positifs brut
top_volume = (
    df
    .orderBy(F.desc("positive"))
    .select("name", "publisher", "positive", "positive_ratio", "release_year")
    .limit(15)
)

display(top_volume)

**Interprétation :** Les deux classements divergent : les jeux au ratio parfait sont souvent des titres de niche très appréciés. En volume, on retrouve les blockbusters comme CS:GO ou Dota 2 avec des millions d'avis.

### Sorties par année — impact du Covid ?

In [ ]:
releases = (
    df
    .filter(F.col("release_year").between(2000, 2024))
    .groupBy("release_year")
    .count()
    .orderBy("release_year")
    .withColumnRenamed("count", "nb_sorties")
)

display(releases)

**Interprétation :** On observe deux accélérations majeures : un x3 entre 2013 et 2014 lié à l'ouverture de Steam Greenlight (2012), qui a drastiquement baissé la barrière d'entrée pour les indépendants, puis une nouvelle hausse à partir de 2017 avec Steam Direct qui a remplacé Greenlight. La période Covid (2020–2021) amplifie encore la tendance avec un pic supplémentaire.

### Distribution des prix et promotions

In [ ]:
free_vs_paid = (
    df
    .withColumn(
        "Type",
        F.when(F.col("price_eur") == 0, "Gratuit (F2P)").otherwise("Payant")
    )
    .groupBy("Type")
    .count()
    .withColumnRenamed("count", "Nombre")
)

display(free_vs_paid)

In [ ]:
price_distribution = (
    df
    .filter(F.col("price_eur") > 0)
    .select("price_eur")
)

display(price_distribution)

In [ ]:
promo_stats = (
    df
    .filter(F.col("price_eur") > 0)
    .withColumn(
        "Statut",
        F.when(F.col("discount") > 0, "En promotion").otherwise("Prix normal")
    )
    .groupBy("Statut")
    .count()
    .withColumnRenamed("count", "Nombre")
)

display(promo_stats)

**Interprétation :** 86% des jeux sont payants avec une médiane à ~6€ — le marché penche vers les petits prix. Les promotions ne concernent qu'environ 5% des jeux au moment du snapshot, mais elles sont une stratégie clé sur Steam.

### Langues les plus représentées

In [ ]:
top_lang = (
    df
    .select(F.explode(F.split(F.col("languages"), ",")).alias("Langue"))
    .withColumn("Langue", F.trim(F.col("Langue")))
    .filter(F.col("Langue") != "")
    .filter(F.col("Langue").isNotNull())
    .groupBy("Langue")
    .count()
    .orderBy(F.desc("count"))
    .limit(15)
    .withColumnRenamed("count", "Nombre de jeux")
)

display(top_lang)

**Interprétation :** L'anglais est présent dans ~99% des jeux. L'allemand, le français et le russe suivent. Le chinois simplifié est 5e — un signal fort de l'importance du marché asiatique pour tout éditeur voulant maximiser sa portée.

### Restrictions d'âge

In [ ]:
age_dist = (
    df
    .withColumn(
        "Tranche",
        F.when(F.col("required_age").isNull(), "Tous publics")
         .when(F.col("required_age") == 0, "Tous publics")
         .when(F.col("required_age") <= 12, "12+")
         .when(F.col("required_age") <= 16, "16+")
         .otherwise("18+")
    )
    .groupBy("Tranche")
    .count()
    .withColumnRenamed("count", "Nombre")
)

display(age_dist)

**Interprétation :** La quasi-totalité des jeux (99%) sont tous publics selon les métadonnées Steam. Les jeux 16+ et 18+ sont peu nombreux mais incluent souvent des titres AAA à fort budget et forte notoriété.

---
## 2. Analyse par Genre

### Genres les plus représentés

In [ ]:
top_genres = (
    df_genres
    .groupBy("genre")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
    .withColumnRenamed("count", "Nombre de jeux")
    .withColumnRenamed("genre", "Genre")
)

display(top_genres)

**Interprétation :** Indie domine massivement (>39k jeux) devant Action et Casual. Ces genres à faible barrière à l'entrée attirent beaucoup de petits studios. Un nouveau titre Ubisoft se positionnerait dans un segment bien plus concurrentiel.

### Ratio d'avis positifs par genre

In [ ]:
genre_reviews = (
    df_genres
    .filter(F.col("total_avis") >= 50)
    .groupBy("genre")
    .agg(
        F.round(F.avg("positive_ratio"), 1).alias("ratio_moyen"),
        F.count("appid").alias("nb_jeux")
    )
    .filter(F.col("nb_jeux") >= 20)
    .orderBy(F.desc("ratio_moyen"))
)

display(genre_reviews)

**Interprétation :** Les genres créatifs (Design, Animation, Audio) ont les meilleurs ratios — leur communauté est de niche mais très satisfaite. À l'inverse, les genres Casual et Free to Play ont les ratios les plus bas, probablement à cause d'attentes plus élevées.

### Genres favoris des top éditeurs

In [ ]:
top10_pub = (
    df
    .groupBy("publisher")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
    .select("publisher")
)

pub_genres = (
    df_genres
    .join(top10_pub, on="publisher", how="inner")
    .groupBy("publisher", "genre")
    .count()
    .withColumnRenamed("count", "nb_jeux")
    .orderBy("publisher", F.desc("nb_jeux"))
)

display(pub_genres)

**Interprétation :** Big Fish Games est concentré sur le Casual — cohérent avec son modèle. SEGA et Square Enix ont des catalogues plus diversifiés (RPG, Action, Adventure). Ce tableau aide Ubisoft à identifier où se positionnent ses concurrents directs.

### Prix moyen et popularité par genre

In [ ]:
genre_revenue = (
    df_genres
    .filter(F.col("price_eur") > 0)
    .groupBy("genre")
    .agg(
        F.round(F.avg("price_eur"), 2).alias("prix_moyen"),
        F.round(F.avg("positive"), 0).alias("avis_positifs_moyens"),
        F.count("appid").alias("nb_jeux")
    )
    .filter(F.col("nb_jeux") >= 20)
    .orderBy(F.desc("avis_positifs_moyens"))
)

display(genre_revenue)

**Interprétation :** Les genres avec un prix élevé ET une forte popularité (quadrant haut-droite) sont les plus lucratifs. Les genres RPG et Action se distinguent sur la popularité. Les outils créatifs (Audio, Video, Animation) ont des prix premium mais une popularité modeste.

---
## 3. Analyse par Plateforme
### Windows / Mac / Linux

In [ ]:
total = df.count()

platform_stats = spark.createDataFrame([
    ("Windows", int(df.filter(F.col("win") == True).count())),
    ("Mac",     int(df.filter(F.col("mac") == True).count())),
    ("Linux",   int(df.filter(F.col("linux") == True).count())),
], ["Plateforme", "Nombre de jeux"]).withColumn(
    "Pourcentage (%)",
    F.round(F.col("Nombre de jeux") / total * 100, 1)
)

display(platform_stats)

In [ ]:
platform_combos = (
    df
    .withColumn("win_str",   F.when(F.col("win"),   F.lit("Win")))
    .withColumn("mac_str",   F.when(F.col("mac"),   F.lit("Mac")))
    .withColumn("linux_str", F.when(F.col("linux"), F.lit("Linux")))
    .withColumn(
        "Combo",
        F.concat_ws(" + ", F.col("win_str"), F.col("mac_str"), F.col("linux_str"))
    )
    .withColumn("Combo", F.when(F.col("Combo") == "", F.lit("Aucune")).otherwise(F.col("Combo")))
    .groupBy("Combo")
    .count()
    .withColumnRenamed("count", "Nombre")
    .orderBy(F.desc("Nombre"))
)

display(platform_combos)

**Interprétation :** Windows est quasi-universel (99.97%). Mac couvre ~23% des jeux et Linux ~15%. La combinaison Win+Mac+Linux représente 12% du catalogue — les studios qui font l'effort du triple portage sont minoritaires mais visibles.

### Genres par plateforme

In [ ]:
genre_by_platform = (
    df_genres
    .groupBy("genre")
    .agg(
        F.round(F.avg(F.col("win").cast(DoubleType())) * 100, 1).alias("pct_win"),
        F.round(F.avg(F.col("mac").cast(DoubleType())) * 100, 1).alias("pct_mac"),
        F.round(F.avg(F.col("linux").cast(DoubleType())) * 100, 1).alias("pct_linux"),
        F.count("appid").alias("nb_jeux")
    )
    .filter(F.col("nb_jeux") >= 20)
    .orderBy(F.desc("pct_mac"))
    .limit(20)
)

display(genre_by_platform)

**Interprétation :** Les genres créatifs (Design, Animation, Audio) et la Stratégie ont les meilleurs taux de portage Mac/Linux. Les gros genres Action et Casual sont quasi-exclusivement Windows. Un jeu de stratégie ou RPG aura une meilleure couverture multi-plateforme naturelle.

---
## Conclusion

| Axe | Enseignement clé |
|---|---|
| **Marché** | 55 000+ jeux, premier boom en 2013–2014 (Steam Greenlight), puis 2017 (Steam Direct) et pic Covid 2020–2021 |
| **Prix** | Médiane à ~6€, 14% de F2P, peu de promos actives au moment du snapshot |
| **Langues** | Anglais indispensable, chinois et russe à prioriser pour l'internationalisation |
| **Genres** | Indie/Casual saturés. RPG et Action dominent en popularité. Stratégie = meilleur ratio qualité/prix |
| **Plateformes** | Windows incontournable. Mac/Linux = avantage différenciant pour Stratégie/RPG |

**Recommandation Ubisoft :** se positionner sur un RPG ou jeu de Stratégie avec pricing ≥ 15€, support anglais + chinois + russe dès le lancement, et portage Windows + Mac pour toucher un public plus large.